# 06 Observability and Telemetry (LiteLLM, 2026)

## What This Lesson Is
Instrument model calls with latency, token, and error telemetry suitable for operations dashboards.

## Scientific Lens
- Concept: Operational observability for LLM systems
- Measure: p95 latency, error rate, and token usage trends
- Validity Limit: Notebook-sized samples are illustrative, not statistically stable baselines.


## How It Works
1. Aggregate synthetic telemetry events.
2. Compute p50/p95 and error rate.
3. Capture live call telemetry in same schema.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Telemetry schema fields:", ["latency_ms", "tokens", "ok", "provider"])


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
events = [
    {"provider": "openai", "latency_ms": 640, "tokens": 450, "ok": True},
    {"provider": "openai", "latency_ms": 730, "tokens": 420, "ok": True},
    {"provider": "openai", "latency_ms": 2100, "tokens": 510, "ok": False},
    {"provider": "ollama", "latency_ms": 1500, "tokens": 380, "ok": True},
]

latencies = sorted(e["latency_ms"] for e in events)
p50 = latencies[len(latencies)//2]
p95 = latencies[min(len(latencies)-1, int(0.95 * len(latencies)))]
error_rate = sum(1 for e in events if not e["ok"]) / len(events)

print({"p50_ms": p50, "p95_ms": p95, "error_rate": round(error_rate, 2)})
assert p95 >= p50


In [ ]:
# Live Demo
import os
import time

try:
    from litellm import completion
except Exception as exc:
    print(f"Skipping live telemetry demo: litellm unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live telemetry demo: OPENAI_API_KEY not set.")
    else:
        t0 = time.perf_counter()
        ok = True
        out = ""
        try:
            r = completion(
                model="openai/gpt-4.1-mini",
                messages=[{"role": "user", "content": "Explain LLM observability in one sentence."}],
                api_key=api_key,
                timeout=20,
            )
            out = r.choices[0].message.content.strip()
            usage = getattr(r, "usage", None)
        except Exception as exc:
            ok = False
            usage = None
            out = str(exc)
        dt = round((time.perf_counter() - t0) * 1000, 1)
        print({"provider": "openai", "latency_ms": dt, "ok": ok, "usage": str(usage)})
        print(out)


## Applied Labs
1. Add per-model aggregation and compare p95 across models.
2. Track error classes (timeout, auth, parse) instead of a single error bucket.
3. Export metrics to a CSV that can be plotted externally.

## Validation Checklist
- Telemetry schema is consistent for deterministic and live paths.
- Latency and success/failure are recorded for every call attempt.
- Metrics can be aggregated without manual post-processing.

## Further Reading
- [OpenTelemetry Concepts](https://opentelemetry.io/docs/concepts/)
- [LiteLLM Observability](https://docs.litellm.ai/docs/observability/integrations)
- [SRE Monitoring Distributed Systems](https://sre.google/sre-book/monitoring-distributed-systems/)
